# Processing raw files

In [10]:
from regime_identification.Data.yf_tickers import load

import pandas as pd
import numpy as np
from datetime import datetime
import os
import yfinance as yf

## .tsv files from [BOLS](https://www.bls.gov/cpi/data.htm)

In [2]:
def process_bols(filename):
    df = pd.read_csv(filename, index_col = None, sep = "\t", 
                     header = None, names = ["ID", "Year", "Month", "Price", "Missing"], 
                     na_values = ["           -"])
    df = df.iloc[:, 1:4] # .dropna(how = "any")
    df["Price"] = df["Price"].astype(float).interpolate() # Fill missing prices with avgs
    df["Month"] = pd.Series([i[1:] for i in df["Month"]]).astype(int)
    # Loop over one month, because this data is only available at the first of the next month
    # df["Month"] = (df["Month"] % 12) + 1
    ## Note: Instead of this, just forward-fill the data. Same result.
    df["Day"] = 1
    df.index = pd.to_datetime(df.loc[:, ["Year", "Month", "Day"]])
    df.index.name = "Date"

    return df["Price"]

bols_df = pd.DataFrame()
for i in os.listdir():
    if not i.endswith(".tsv"): continue
        
    name = i[:-4]
    bols_df[name] = process_bols(i)

bols_df.tail(40)

,electricity_kwh,gasoline_gallon,household-gas_therm,housing,inflation_cpi
Date,,,,,
2023-04-01,0.1650,3.839,1.413,317.366,302.845
2023-05-01,0.1650,3.794,1.385,318.146,303.334
2023-06-01,0.1700,3.821,1.371,319.355,304.014
2023-07-01,0.1690,3.842,1.395,320.552,304.609
2023-08-01,0.1700,4.064,1.402,321.520,306.082
2023-09-01,0.1710,4.107,1.377,323.250,307.276
2023-10-01,0.1690,3.910,1.388,324.251,307.696
2023-11-01,0.1680,3.623,1.442,325.495,308.148
2023-12-01,0.1690,3.411,1.429,326.520,308.741


In [3]:
# saving
bols_df.to_csv("bols.csv")

## 3M-10Y Treasury Yield Curve

In [6]:
t_yields = pd.DataFrame()
t_files = ["treasury_yield_3mo.csv", "treasury_yield_10y.csv"]
for filename in t_files:
    df = pd.read_csv(filename, index_col = 0, 
                     header = 0, names = ["Date", "Price"], 
                    )
    name = filename[:-4]
    t_yields[name] = df

# Yield curve
t_yields["treasury_yield_curve"] = t_yields["treasury_yield_3mo"] / t_yields["treasury_yield_10y"]

t_yields.to_csv("treasury_yields.csv")

In [7]:
t_yields

,treasury_yield_3mo,treasury_yield_10y,treasury_yield_curve
Date,,,
1981-09-01,17.01,15.41,1.103829
1981-09-02,16.65,15.40,1.081169
1981-09-03,16.96,15.48,1.095607
1981-09-04,16.64,15.51,1.072856
1981-09-07,NaN,NaN,NaN
...,...,...,...
2026-08-26,3.85,4.66,0.826180
2026-08-27,3.84,4.67,0.822270
2026-08-28,3.90,4.73,0.824524


## Foreign Exchange

In [8]:
fx_df = pd.DataFrame()
fx_files = ["fx_JPUS.csv", "fx_USEU.csv", "fx_USUK.csv"]
for filename in fx_files:
    df = pd.read_csv(filename, index_col = 0, 
                     header = 0, names = ["Date", "Price"], 
                    )
    name = filename[:-4]
    fx_df[name] = df

fx_df.to_csv("fx_rates.csv")

## Gold

In [28]:
gold = pd.read_csv("gold_historical.csv", index_col = 0, names = ["Date", "gold", "Source"])
gold = gold.drop("Source", axis = 1).iloc[1:, :]
gold = gold[~gold.index.duplicated(keep = "first")]

In [29]:
gold.to_csv("gold.csv")

## VIX

In [16]:
vix = yf.Ticker("^VIX").history(period = "50y")
vix = pd.DataFrame(vix["Close"], index = vix.index)
vix.columns = ["VIX"]

In [24]:
vix.to_csv("VIX.csv")

# Assembly

In [25]:
polished_csvs = [ 
    "bols.csv", "interest_rates.csv", 
    "sp500_avg-pe-ratio.csv", "treasury_yields.csv",
    "US_GDP.csv", "fx_rates.csv",
    "gold.csv", "VIX.csv"
]

dfl = []
for i in polished_csvs:
    dfl.append(load(i))

final = pd.concat(dfl, axis = 1, join = "outer", ignore_index = False, sort = True)

final.to_csv("extramarket.csv")
final

,electricity_kwh,gasoline_gallon,household-gas_therm,housing,inflation_cpi,BLP,FF,pe_ratio,treasury_yield_3mo,treasury_yield_10y,treasury_yield_curve,GDP,fx_JPUS,fx_USEU,fx_USUK,gold,VIX
Date,,,,,,,,,,,,,,,,,
1258-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.89,NaN
1259-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.89,NaN
1260-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.89,NaN
1261-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.89,NaN
1262-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.89,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.300000
2026-09-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.720000
2026-09-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.459999


In [27]:
final.index.has_duplicates

False